In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib agg
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
       
from HPIB.HP4155 import HP4155
from HPIB.HPT import Plot, PlotVgs, PlotVp, CalcIsSat, SecDer, Plot2P, PlotDiode4P
from HPIB.DevParams import UMC
from HPIB.INOSerial import Arduino

from BFmodule import DR

from YFunc import YFuncExtraction

from Test import TestDevice, WriteLog

from IPython.display import clear_output, display
from os import makedirs, rename
from time import sleep
from datetime import datetime, timedelta

Elsa=DR()

HP=HP4155("GPIB0::17", debug=False)
HP.IntTime="LONG"
HP.reset()

INO=Arduino("COM3")
print(INO.ask('*'.encode()))

Elsa v2.4.2
HEWLETT-PACKARD,4155A,0,01.04:01.04:01.00
InoMatrix



In [2]:
def WriteLog(msg, path, mode='a', end='\n', output=True):
    with open(path, mode) as logfile:
        logfile.write(msg+end)
    if output:
        print(f"{msg}{end}", end='')

def TestDevice(device, chn, path, params, HiPot=False):
    global HP, INO, Elsa, prog_bar
    INO.opench(chn+1)

    if not device:
        INO.opench(0)
        sleep(2)
        return 0

    if device[:2].upper() not in ['CA', 'CB', 'CG', 'TP', 'TN', 'DP', 'DN']:
        return "Invalid device"

    WriteLog(f"## Ch {chn+1} {device}", path + 'log.txt')
    pathp=path+device
    makedirs(pathp, exist_ok=True)
    
    ####################### Measure Diode
    #
    #
    if 'D' in device.upper():
        
        # HP.StopCond="COMP"
        HP.IntTime="MED"
        HP.Diode4P(params['Vfmin'], params['Vfmax'], params['Vfstep'], Comp=params['IComp'] if not HiPot else 2*params['IComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M%S')
        V100, V10=PlotDiode4P(HP.SingleSave(f"{pathp}/{now}.csv", timeout=30, real=True))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/{now}.csv", f"{pathp}/Diode - {temp} - {now}.csv")
            rename(f"{pathp}/{now}.png", f"{pathp}/Diode - {temp} - {now}.png")
            rename(f"{pathp}/{now} log.png", f"{pathp}/Diode - {temp} - {now} log.png")
            
            with open(f"{pathp}/V10010uA.log", 'a') as DiodeParam:
                DiodeParam.write(f"{temp},{format(V100, '.3f')},{format(V10, '.3f')}\n")
        except Exception as err:
            print (">>> Error:", err)
        
        now=datetime.now()
            
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")
            
        HP.StopCond="OFF"
        HP.IntTime="LONG"
        
        INO.opench(0)
        WriteLog('', path + 'log.txt')
        return 0
    #
    #
    #######################

    ####################### Measure Transistor
    #
    #

    if 'T' in device.upper():
        
        ptype='P' in device.upper()
        
        HP.SetVgs(params['Vmin'], params['Vmax'], params['Vstep'], params['Vd'], ptype=ptype)
  
        now=datetime.now().strftime('%y%m%d %H%M')
        LIN = PlotVgs(HP.SingleSave(f"{pathp}/IdVgs - {now}.csv", timeout=30))
        
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            newname=f"{pathp}/IdVgs - {temp} - {now}"
            rename(f"{pathp}/IdVgs - {now}.csv", f"{newname}.csv")
            rename(f"{pathp}/IdVgs - {now}.png", f"{newname}.png")
        except Exception as err:
            print (">>> Error:", err)
        now=datetime.now()

        try:
            LIN, Vth, SS, migm, miyf, theta1, theta2, errmax = YFuncExtraction(f"{newname}.csv", UMC[int(device[2:])], 4.2, 3.9, params['Vd'])
            WriteLog(f"LIN={format(LIN, '.3f')}, Vth={format(Vth, '.3f')} V, SS={format(SS, '.2f')} mV/dec, miyf={format(migm, '.1f')}, miyf={format(miyf, '.1f')}, theta1={format(theta1, '.3e')}, theta2={format(theta2, '.3e')}", path + 'log.txt')
            WriteLog(f"{temp},{format(LIN, '.3f')},{format(Vth, '.3f')},{format(SS, '.2f')},{format(migm, '.1f')},{format(miyf, '.1f')},{format(theta1, '.3e')},{format(theta2, '.3e')}", pathp + '/params.txt')
        except Exception as err:
            print (">>> Error:", err)
            WriteLog(f"Vth={LIN} V", path + 'log.txt')
        
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")
            
        if HiPot:
            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVgs(params['Vmin'], params['Vmax'], params['Vstep'], params['Vmax'], ptype=ptype, sat=True)
            HP.SingleSave(f"{pathp}/IdVgsSatUp - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVgsSatUp - {now}.csv", f"{pathp}/IdVgsSatUp - {temp} - {now}.csv")
            except Exception as err:
                print(">>> Error:", err)
            n1, Ispec1 = CalcIsSat(f"{pathp}/IdVgsSatUp - {temp} - {now}.csv",temp)
            WriteLog(f"n={format(n, '.3f')}, Ispec={format(Ispec, '.3e')} A", path + 'log.txt')

            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")

            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVgs(-params['Vmin'], -params['Vmax'], -params['Vstep'], params['Vmax'], ptype=ptype, sat=True)
            HP.SingleSave(f"{pathp}/IdVgsSatDown - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVgsSatDown - {now}.csv", f"{pathp}/IdVgsSatDown - {temp} - {now}.csv")
            except Exception as err:
                print(">>> Error:", err)
            n2, Ispec2 = CalcIsSat(f"{pathp}/IdVgsSatDown - {temp} - {now}.csv",temp)
            WriteLog(f"n={format(n, '.3f')}, Ispec={format(Ispec, '.3e')} A", path + 'log.txt')

            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")
            
            if Ispec1 != 0:
                now=datetime.now().strftime('%y%m%d %H%M')
                HP.SetVp(Ispec, params['Vmin'], params['Vmax'], 0.05, ptype=ptype)
                HP.SingleSave(f"{pathp}/VpVg - {now}.csv", timeout=30)
                try:
                    temp=format(Elsa.GetT('t4k'), '07.3f')
                    rename(f"{pathp}/VpVg - {now}.csv", f"{pathp}/VpVg - {temp} - {now}.csv")
                except Exception as err:
                    print (">>> Error:", err)
                VTO=PlotVp(f"{pathp}/VpVg - {temp} - {now}.csv")
                WriteLog(f"VTO={VTO} V", path + 'log.txt')

                now=datetime.now()
                while (datetime.now()-now).seconds < 4*params['min_wait']:
                    prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                    sleep(0.5)
                prog_bar.update("Measuring")

            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVds(params['Vmin'], params['Vmax'], params['Vstep'], params['Vgmin'], params['Vgmax'], params['Vgstep'], ptype=ptype)
            HP.SingleSave(f"{pathp}/IdVdsUp - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVdsUp - {now}.csv", f"{pathp}/IdVdsUp - {temp} - {now}.csv")
            except Exception as err:
                print (">>> Error:", err)
            Plot(f"{pathp}/IdVdsUp - {temp} - {now}.csv", 'Vd', 'Id')
            
            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")

            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVds(-params['Vmin'], -params['Vmax'], -params['Vstep'], params['Vgmin'], params['Vgmax'], params['Vgstep'], ptype=ptype)
            HP.SingleSave(f"{pathp}/IdVdsDown - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVdsDown - {now}.csv", f"{pathp}/IdVdsDown - {temp} - {now}.csv")
            except Exception as err:
                print (">>> Error:", err)
            Plot(f"{pathp}/IdVdsDown - {temp} - {now}.csv", 'Vd', 'Id')
            
            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")
            
        INO.opench(0)
        WriteLog('', path + 'log.txt')
        return 0
    #
    #
    #######################
        
    ####################### Measure CrossBridge
    #
    #
    if 'CG' in device.upper():
           
        now=datetime.now().strftime('%y%m%d %H%M')
        V10u=HP.MeasV10u(f"{pathp}/2P - {now}.csv", timeout=0.5)
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/2P - {now}.csv", f"{pathp}/2P - {temp} - {now}.csv")
            rename(f"{pathp}/2P - {now}.png", f"{pathp}/2P - {temp} - {now}.png")
        except Exception as err:
            print (">>> Error:", err)
        WriteLog(f"V_10u={format(V10u, '.2f')}", path + 'log.txt')
        with open(f"{pathp}/VxT 10uA.log", 'a') as VxT:
            VxT.write(f"{temp},{format(V10u, '.4e')}\n")

    if 'CA' in device.upper():

        HP.Set2P(params['Ifmin']/params['Iffactor'], params['Ifmax']/params['Iffactor'], params['Ifpoints'], SMUN='SMU1', SMUP='SMU2', Comp=params['VComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M')
        Rshort=Plot2P(HP.SingleSave(f"{pathp}/2P - {now}.csv", timeout=30))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/2P - {now}.csv", f"{pathp}/2P - {temp} - {now}.csv")
            rename(f"{pathp}/2P - {now}.png", f"{pathp}/2P - {temp} - {now}.png")
        except Exception as err:
            print (">>> Error:", err)
        WriteLog(f"Rshort={format(Rshort, '.2f')}", path + 'log.txt')
        with open(f"{pathp}/RxT 2P.log", 'a') as RxT2p:
            RxT2p.write(f"{temp},{format(Rshort, '.2f')}\n")

    if 'CB' in device.upper():
    
        HP.Set4P(params['Ifmin'], params['Ifmax'], params['Ifpoints'])
        
        now=datetime.now().strftime('%y%m%d %H%M')
        RCB=Plot2P(HP.SingleSave(f"{pathp}/4P - {now}.csv", timeout=30))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/4P - {now}.csv", f"{pathp}/4P - {temp} - {now}.csv")
            rename(f"{pathp}/4P - {now}.png", f"{pathp}/4P - {temp} - {now}.png")
        except Exception as err:
            print (">>> Error:", err)
        WriteLog(f"RCB={format(RCB, '.2e')}", path + 'log.txt')
        with open(f"{pathp}/RxT 4P.log", 'a') as RxT4p:
            RxT4p.write(f"{temp},{format(RCB, '.2e')}\n")

    now=datetime.now()
    while (datetime.now()-now).seconds < params['min_wait']:
        prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
        sleep(0.5)
    prog_bar.update("Measuring")
            
    INO.opench(0)
    WriteLog('', path + 'log.txt')
    return 0
    #
    #
    #######################

In [5]:
params = {
'Vd' : 0.05,
'Vmin' : 0,
'Vmax' : 1.5,
'Vstep' : 0.02,

'Vgmin' : 0.6,
'Vgmax' : 1.4,
'Vgstep' : 0.2,

'Vfmin' : 0,
'Vfmax' : 1.5,
'Vfstep' : 0.02,
'IComp' : 1e-3,

'Ifmin' : -1e-3,
'Ifmax' : 1e-3,
'Ifpoints' : 50,
'Iffactor' : 50,
'VComp' : 1.5,

'min_wait' : 15
}

DeviceList = ['CB1']
# DeviceList = ['', 'DN1']
prepath = "C:/Users/Zucchi/Documents/Medidas/250205 CB1/"

In [4]:
INO.opench(1)

'Open INO: ch 1'

In [7]:
HP.Set4P(params['Ifmin'], params['Ifmax'], params['Ifpoints'])

Set 4P
I=(-0.001, 0.001), 50 Points


0

In [8]:
now=datetime.now().strftime('%y%m%d %H%M')
RCB=Plot2P(HP.SingleSave(f"{prepath}/4P - {now}.csv", timeout=1))
try:
    temp=format(Elsa.GetT('t4k'), '07.3f')
    rename(f"{prepath}/4P - {now}.csv", f"{prepath}/4P - {temp} - {now}.csv")
    rename(f"{prepath}/4P - {now}.png", f"{prepath}/4P - {temp} - {now}.png")
except Exception as err:
    print (">>> Error:", err)

Done 4P. Duration: 6 s                                     


In [17]:
## test measurement

try:
    current_temp=Elsa.GetT('t4k')
except:
pass

path = prepath
makedirs(path, exist_ok=True)

MsrNo=1 ## Take {MsrNo} measurements
MsrWait=1 ## wait {MsrWait} minutes

HP.IntTime='LONG'
HP.LongNPLC=4

prog_bar=display('',display_id=True)

for i in range(MsrNo):
    start=datetime.now()
    try:
        current_temp=Elsa.GetT('t4k')
    except:
        current_temp='Fail'
    Measurement_Msg=f"#  Measurement {i+1} - {current_temp} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('Measuring',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params)

    clear_output()
    prog_bar=display('Measuring',display_id=True)
    print(Measurement_Msg)
    
    try:
        current_temp=Elsa.GetT('t4k')
    except:
        current_temp='Fail'
    WriteLog(f"Measurement {i+1} end - {current_temp}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    if i < MsrNo-1:
        start=datetime.now()
        while (datetime.now()-start).seconds < MsrWait*60:
            prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
            sleep(1)    
prog_bar.update("Pre measurement done")

'Pre measurement done'

In [ ]:
## Pre measurement

try:
    current_temp=Elsa.GetT('t4k')
except:
    pass

if current_temp > 150:
    path = prepath+"PreCool/"
else:
    path = prepath+"Cold/"
makedirs(path, exist_ok=True)

MsrNo=3 ## Take {MsrNo} measurements
MsrWait=5 ## wait {MsrWait} minutes

HP.IntTime='LONG'
HP.LongNPLC=2

prog_bar=display('',display_id=True)

for i in range(MsrNo):
    start=datetime.now()
    try:
        current_temp=Elsa.GetT('t4k')
    except:
        current_temp='Fail'
    Measurement_Msg=f"#  Measurement {i+1} - {current_temp} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('Measuring',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params, True)

    clear_output()
    prog_bar=display('Measuring',display_id=True)
    print(Measurement_Msg)
    TestDevice('DN1', 0, path, params)
    
    try:
        current_temp=Elsa.GetT('t4k')
    except:
        current_temp='Fail'
    WriteLog(f"Measurement {i+1} end - {current_temp}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    if i < MsrNo-1:
        start=datetime.now()
        while (datetime.now()-start).seconds < MsrWait*60:
            prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
            sleep(1)    
prog_bar.update("Pre measurement done")

'Measuring'

#  Measurement 1 - 270.721 - 250206 0938

## Ch 1 CB1
Set 4P
I=(-0.005, 0.005), 50 Points
Measuring 4P   15s 15s 15s 15s | 15s 15s 15s

In [5]:
###### Vac measurement

P6=500

prog_bar=display('',display_id=True)

while P6 < 0.9 and P6 >1e-3:
    try:
        P6=Elsa.GetPCh(6)
    except:
        pass
    prog_bar.update(f"{datetime.now().strftime('%H:%M:%S')} - P6 = {format(P6, '.1f')} mBarr")
    sleep(1)

try:
    current_temp=Elsa.GetT('t4k')
except:
    pass

if current_temp > 150:
    path = prepath+"Vac/"
else:
    path = prepath+"Cold/"
makedirs(path, exist_ok=True)

MsrNo=3 ## Take {MsrNo} measurements
MsrWait=1 ## wait {MsrWait} minutes

HP.IntTime='LONG'
HP.LongNPLC=10

for i in range(MsrNo):
    start=datetime.now()
    try:
        current_temp=Elsa.GetT('t4k')
    except:
        current_temp='Fail'
    Measurement_Msg=f"#  Measurement {i+1} - {current_temp} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('Measuring',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params, True)
    try:
        current_temp=Elsa.GetT('t4k')
    except:
        current_temp='Fail'
    WriteLog(f"Measurement {i+1} end - {current_temp}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    if i < (MsrNo-1):
        start=datetime.now()
        while (datetime.now()-start).seconds < MsrWait*60:
            prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
            sleep(1)
    
prog_bar.update("Vac measurement done")

'Vac measurement done'

In [18]:
###### Warmup measurement

try:
    current_temp=Elsa.GetT('t4k')
except:
    pass

if current_temp > 150:
    path = prepath+"Cooldown/"
else:
    path = prepath+"Warmup/"
makedirs(path, exist_ok=True)

freq_temp=10
MsrNo=3 ## Take {MsrNo} measurements
MsrWait=0.5 ## wait {MsrWait} minutes
# failsafe_time=[60, 120, 120, 60] ## Maximum wait time between measurements (API crash failsafe)
# failsafe_temp=[0, 120, 180, 300] ## is fitted from these temps and times
# failsafe=np.polyfit(failsafe_time, failsafe_temp, 2)
failsafe=[-2.4e-03, 4.3e-01, 1.3e+02]

HP.IntTime='LONG'
HP.LongNPLC=2

clear_output()

prog_bar=display('',display_id=True)

try:
    current_temp=Elsa.GetT('t4k')
except:
    pass

if current_temp > 150: finish_temp=5
else: finish_temp=295

last_temp=150

while (last_temp != finish_temp):
    #loop until temp changes and is a multiple of freq_temp
    MaxWait=np.polyval(failsafe, last_temp) ## API crash failsafe calculation
    start=datetime.now()
    while (np.around(current_temp)==last_temp or np.around(current_temp)%freq_temp != 5):
        if (datetime.now()-start).seconds/60 > MaxWait: ## Failsafe trigger
            break
        for i in range(15):
            prog_bar.update(f"{datetime.now().strftime('%H:%M:%S')} - T = {format(current_temp, '.1f')} K")
            sleep(1)
    
        current_temp=Elsa.GetT('t4k')

    last_temp=np.around(current_temp)
    
    for Msr in range(MsrNo):
        plt.close('all')
        clear_output()
        prog_bar=display('Measuring',display_id=True)

        start=datetime.now()
        WriteLog(f"# {current_temp} K - Measurement {Msr+1} - {datetime.now().strftime('%y%m%d %H%M')}\n", path+'log.txt')
        for chn, device in enumerate(DeviceList):
            if device:
                TestDevice(device, chn, path, params)

        TestDevice('DN1', 0, path, params)
    
        WriteLog(f"{current_temp} K - Measurement {Msr+1} end . Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    try:
        current_temp=Elsa.GetT('t4k')
    except:
        pass
        if i < MsrNo:
            start=datetime.now()
            while (datetime.now()-start).seconds < MsrWait*60:
                prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
                sleep(1)
prog_bar.update("Ramp measurement done")

'Measuring'

# 295.411 K - Measurement 1 - 250205 2030

## Ch 1 CB1
Set 4P
I=(-0.005, 0.005), 50 Points
Measuring 4P + 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s 15s | 15s 15s 15s
Error opening CSV



UnboundLocalError: cannot access local variable 'df' where it is not associated with a value

In [ ]:
###### 1 hour wait

prog_bar=display('',display_id=True)

start=datetime.now()
while (datetime.now()-start).seconds < 3600:
    prog_bar.update(f"Waiting: {3600-(datetime.now()-start).seconds} s")
    sleep(1)

In [38]:
###### Post measurement

if Elsa.GetT('t4k') > 150:
    path = prepath+"PostCool/"
else:
    path = prepath+"Cold/"
makedirs(path, exist_ok=True)

MsrNo=3 ## Take {MsrNo} measurements
MsrWait=5 ## wait {MsrWait} minutes

HP.IntTime='LONG'
HP.LongNPLC=10

for i in range(MsrNo):
    start=datetime.now()
    Measurement_Msg=f"#  Measurement {i+1} - {Elsa.GetT('t4k')} - {start.strftime('%y%m%d %H%M')}\n"
    WriteLog(Measurement_Msg, path+'log.txt')
    
    for chn, device in enumerate(DeviceList):
        clear_output()
        prog_bar=display('',display_id=True)
        print(Measurement_Msg)
        if device:
            TestDevice(device, chn, path, params, True)

        clear_output()
        prog_bar=display('Measuring',display_id=True)
        print(Measurement_Msg)
        TestDevice('DN1', 0, path, params)
    
    WriteLog(f"Measurement {i+1} end - {Elsa.GetT('t4k')}. Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    clear_output()
    plt.close('all')
    prog_bar=display('',display_id=True)
    if i < MsrNo-1:
        start=datetime.now()
        while (datetime.now()-start).seconds < MsrWait*60:
            prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
            sleep(1)    

print("Post measurement done")

''

Post measurement done


In [35]:
HP.close()
INO.close()
print("Comm closed")

Comm closed
